# Swin-UMamba trên Kaggle — Train / Val / Inference

Notebook này chạy lại pipeline của bài gốc *Swin-UMamba* (nnU-Net v2, cấu hình **2d**), nhưng dùng bộ dữ liệu mới.

**Trước khi chạy, cấu hình Kaggle:**
1. *Settings → Accelerator*: **GPU T4** (dùng 1 GPU). Không nên dùng P100 vì torch mới trên Kaggle không còn hỗ trợ tốt.
2. *Settings → Internet*: **On** (để clone GitHub, cài `mamba-ssm` và tải pretrained VMamba).
3. *Add Input*: thêm Kaggle Dataset chứa thư mục **BTXRD** (gồm `images/` và `Annotations/`). Notebook sẽ tự tìm thư mục này trong `/kaggle/input`.
4. Nếu train lâu hơn 12 giờ: dùng **Save Version → Save & Run All (Commit)**. Kết quả trong `/kaggle/working` sẽ được lưu lại. Ở phiên sau, thêm output đó làm input và đặt `RESUME_DIR` để train tiếp.

Các bước: clone repo → cài môi trường → tải pretrained → chuyển dữ liệu sang định dạng nnU-Net → `nnUNetv2_plan_and_preprocess` → `nnUNetv2_train` (kèm validation) → `nnUNetv2_predict` trên tập test → tính DSC / NSD → trực quan hoá.

## 0. Cấu hình

In [ ]:
import os, glob

# ---------- Mã nguồn ----------
REPO_URL = "https://github.com/67htb5vn/Swin-UMamba.git"
BRANCH   = "main"
WORK     = "/kaggle/working"
REPO     = f"{WORK}/Swin-UMamba"
TMP      = "/tmp/nnunet"            # raw + preprocessed để ở /tmp (đĩa lớn, không bị lưu vào output)

# ---------- Dữ liệu ----------
# "btxrd"     : chuyển BTXRD (LabelMe json) sang nnU-Net
# "nnunet_raw": dùng sẵn một thư mục DatasetXXX_* đúng định dạng nnU-Net (vd. dữ liệu gốc của U-Mamba)
DATA_SOURCE  = "btxrd"
BTXRD_DIR    = None                 # None = tự tìm trong /kaggle/input; hoặc vd "/kaggle/input/btxrd/BTXRD"
NNUNET_RAW_DIR = None               # dùng khi DATA_SOURCE="nnunet_raw", vd "/kaggle/input/umamba-data/Dataset704_Endovis17"
DATASET_ID   = 705
DATASET_NAME = f"Dataset{DATASET_ID:03d}_BTXRD"
LABEL_MODE   = "binary"             # "binary": nền/khối u | "benign_malignant": nền/lành tính/ác tính
MAX_SIZE     = 1024                 # resize cạnh dài nhất về <= MAX_SIZE (ảnh X-quang gốc rất lớn); None = giữ nguyên
TEST_RATIO   = 0.2                  # tỉ lệ ảnh để làm tập test (imagesTs/labelsTs)
SEED         = 42

# ---------- Huấn luyện ----------
# Trainer dạng *Kaggle = trainer gốc + cho phép chỉnh epoch/iter qua biến môi trường (xem bước 4)
#   nnUNetTrainerSwinUMambaKaggle        : Swin-UMamba, pretrain ImageNet
#   nnUNetTrainerSwinUMambaDKaggle       : Swin-UMamba†, pretrain ImageNet
#   nnUNetTrainerSwinUMambaScratchKaggle : Swin-UMamba, không pretrain
#   nnUNetTrainerSwinUMambaDScratchKaggle: Swin-UMamba†, không pretrain
TRAINER         = "nnUNetTrainerSwinUMambaKaggle"
FOLD            = "0"               # "0".."4": train 80% / val 20% (có validation thật). "all": giống script gốc (train trên toàn bộ)
EPOCHS          = 100               # code gốc mặc định 1000 epoch -> quá dài cho Kaggle
ITERS_PER_EPOCH = 250
SAVE_EVERY      = 10                # lưu checkpoint_latest mỗi N epoch (để resume)
BATCH_SIZE      = None              # None = để nnU-Net tự chọn; giảm (vd 4) nếu bị CUDA OOM
RESUME_DIR      = None              # vd "/kaggle/input/<output-phien-truoc>/Swin-UMamba/data/nnUNet_results"

PRETRAIN_URL = "https://github.com/MzeroMiko/VMamba/releases/download/%2320240218/vssmtiny_dp01_ckpt_epoch_292.pth"

# ---------- Biến môi trường (được kế thừa bởi các lệnh `!`) ----------
os.environ.update({
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",   # torch>=2.6 mặc định weights_only=True -> không load được checkpoint nnU-Net
    "nnUNet_n_proc_DA": "4",                   # Kaggle có 4 CPU
    "nnUNet_compile": "f",
    "MPLBACKEND": "Agg",
    "SUM_EPOCHS": str(EPOCHS),
    "SUM_ITERS": str(ITERS_PER_EPOCH),
    "SUM_SAVE_EVERY": str(SAVE_EVERY),
})

## 1. Kiểm tra GPU & clone repo từ GitHub

In [ ]:
!nvidia-smi
import subprocess
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=False)
!git -C {REPO} log --oneline -1

## 2. Cài môi trường
`mamba-ssm` sẽ tự tải wheel build sẵn khớp với torch/CUDA của Kaggle nếu có. Nếu không có wheel phù hợp, nó sẽ build từ source (mất khoảng 20–40 phút).

In [ ]:
%cd {REPO}/swin_umamba
!pip install -q -e .
# setup.py ghim monai==1.3.0 (không tương thích numpy 2 trên Kaggle) -> dùng 1.4.0, API các block vẫn giữ nguyên
!pip install -q "monai==1.4.0" timm torchinfo einops numba packaging ninja
!pip install -q --no-build-isolation causal-conv1d mamba-ssm
%cd {REPO}

In [ ]:
# Kiểm tra kernel CUDA selective_scan của mamba
check = ("import torch, mamba_ssm; "
         "from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; "
         "u=torch.randn(1,4,16,device='cuda'); d=torch.randn(1,4,16,device='cuda'); A=-torch.rand(4,8,device='cuda'); "
         "B=torch.randn(1,8,16,device='cuda'); C=torch.randn(1,8,16,device='cuda'); "
         "print('torch', torch.__version__, '| mamba_ssm', mamba_ssm.__version__, '| out', selective_scan_fn(u,d,A,B,C).shape)")
subprocess.run(["python", "-c", check], check=True)
!python -c "import nnunetv2, monai; from nnunetv2.paths import nnUNet_raw; print('nnUNet_raw =', nnUNet_raw)"

## 3. Tải pretrained VMamba-Tiny (ImageNet) & chuẩn bị thư mục dữ liệu

In [ ]:
os.chdir(REPO)
ckpt = f"{REPO}/data/pretrained/vmamba/vmamba_tiny_e292.pth"
os.makedirs(os.path.dirname(ckpt), exist_ok=True)
if not os.path.isfile(ckpt):
    !wget -q -O {ckpt} "{PRETRAIN_URL}"
print("pretrained:", ckpt, os.path.getsize(ckpt) // 2**20, "MB")

# raw & preprocessed nằm ở /tmp (symlink vào repo/data), results nằm trong /kaggle/working (được lưu lại)
RAW_DS = f"{TMP}/nnUNet_raw/{DATASET_NAME}"
os.makedirs(RAW_DS, exist_ok=True)
os.makedirs(f"{TMP}/nnUNet_preprocessed", exist_ok=True)
os.makedirs(f"{REPO}/data/nnUNet_results", exist_ok=True)
for link, target in [(f"{REPO}/data/nnUNet_raw/{DATASET_NAME}", RAW_DS),
                     (f"{REPO}/data/nnUNet_preprocessed", f"{TMP}/nnUNet_preprocessed")]:
    if not os.path.lexists(link):
        os.symlink(target, link)

if RESUME_DIR:   # khôi phục kết quả phiên trước để train tiếp
    !cp -rn {RESUME_DIR}/. {REPO}/data/nnUNet_results/
    print("restored results from", RESUME_DIR)

## 4. Trainer cho Kaggle
Các trainer này kế thừa nguyên trainer gốc (cùng AdamW, lr 1e-4, cosine schedule, đóng băng encoder 10 epoch đầu, deep supervision), chỉ thay đổi:
- `num_epochs`, `num_iterations_per_epoch`, `save_every`: đọc từ biến môi trường;
- không lưu `checkpoint_{epoch}.pth` (mỗi file khoảng 0.5 GB, dễ làm đầy 20 GB của `/kaggle/working`). Vẫn giữ `checkpoint_latest`, `checkpoint_best` và `checkpoint_final`.

In [ ]:
trainer_src = '''
import os
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMamba import nnUNetTrainerSwinUMamba
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaD import nnUNetTrainerSwinUMambaD
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaScratch import nnUNetTrainerSwinUMambaScratch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaDScratch import nnUNetTrainerSwinUMambaDScratch


def _kaggle_setup(self):
    self.num_epochs = int(os.environ.get("SUM_EPOCHS", 100))
    self.num_iterations_per_epoch = int(os.environ.get("SUM_ITERS", 250))
    self.save_every = int(os.environ.get("SUM_SAVE_EVERY", 10))

'''
for base in ["nnUNetTrainerSwinUMamba", "nnUNetTrainerSwinUMambaD",
             "nnUNetTrainerSwinUMambaScratch", "nnUNetTrainerSwinUMambaDScratch"]:
    trainer_src += f'''
class {base}Kaggle({base}):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 unpack_dataset: bool = True, device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)
        _kaggle_setup(self)

    def on_epoch_end(self):
        nnUNetTrainer.on_epoch_end(self)   # bỏ qua việc lưu checkpoint_{{epoch}}.pth của lớp cha

'''
path = f"{REPO}/swin_umamba/nnunetv2/training/nnUNetTrainer/nnUNetTrainerSwinUMambaKaggle.py"
open(path, "w").write(trainer_src)
!python -c "from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaKaggle import *; print('OK')"

## 5. Chuẩn bị dữ liệu theo định dạng nnU-Net
**BTXRD:** chỉ dùng các ảnh có nhãn **polygon** (vùng khối u); bỏ qua các `rectangle` (bbox). Ảnh được chuyển sang PNG xám 1 kênh, mask được lưu dạng PNG với giá trị 0/1 (hoặc 0/1/2), và dữ liệu được chia train/test theo `TEST_RATIO`.

In [ ]:
convert_src = r"""
import os, sys, json, glob, random
import numpy as np, cv2
from PIL import Image, ImageOps

src, out, mode, max_size, test_ratio, seed, name = sys.argv[1:8]
max_size = int(max_size) if max_size != "None" else None
test_ratio, seed = float(test_ratio), int(seed)

MALIGNANT = {"osteosarcoma", "other mt"}
if mode == "binary":
    labels = {"background": 0, "tumor": 1}
    cls = lambda l: 1
else:
    labels = {"background": 0, "benign": 1, "malignant": 2}
    cls = lambda l: 2 if l.strip().lower() in MALIGNANT else 1

imgs = {os.path.splitext(f)[0]: os.path.join(src, "images", f) for f in os.listdir(os.path.join(src, "images"))}
cases = []
for jf in sorted(glob.glob(os.path.join(src, "Annotations", "*.json"))):
    stem = os.path.splitext(os.path.basename(jf))[0]
    ann = json.load(open(jf))
    polys = [s for s in ann["shapes"] if s.get("shape_type") == "polygon" and len(s["points"]) >= 3]
    if stem in imgs and polys:
        cases.append((stem, imgs[stem], ann, polys))
print("annotated cases with polygon:", len(cases))

random.Random(seed).shuffle(cases)
n_test = int(round(len(cases) * test_ratio))
splits = {"Ts": cases[:n_test], "Tr": cases[n_test:]}
for d in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    os.makedirs(os.path.join(out, d), exist_ok=True)

for split, items in splits.items():
    for stem, ip, ann, polys in items:
        img = ImageOps.exif_transpose(Image.open(ip)).convert("L")   # LabelMe cũng áp dụng EXIF orientation
        w, h = img.size
        sx = w / float(ann.get("imageWidth") or w)
        sy = h / float(ann.get("imageHeight") or h)
        mask = np.zeros((h, w), np.uint8)
        for c in (1, 2):   # vẽ ác tính sau để ưu tiên khi chồng lấn
            for s in polys:
                if cls(s["label"]) == c:
                    pts = np.array([[x * sx, y * sy] for x, y in s["points"]], np.float64).round().astype(np.int32)
                    cv2.fillPoly(mask, [pts], c)
        img = np.array(img)
        if max_size and max(h, w) > max_size:
            r = max_size / max(h, w)
            nw, nh = int(round(w * r)), int(round(h * r))
            img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
            mask = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(os.path.join(out, f"images{split}", f"{stem}_0000.png"), img)
        cv2.imwrite(os.path.join(out, f"labels{split}", f"{stem}.png"), mask)

json.dump({"channel_names": {"0": "X-ray"}, "labels": labels, "numTraining": len(splits["Tr"]),
           "file_ending": ".png", "name": name,
           "description": "BTXRD bone tumor X-ray, polygons -> segmentation masks"},
          open(os.path.join(out, "dataset.json"), "w"), indent=2)
print("train:", len(splits["Tr"]), "| test:", len(splits["Ts"]), "| labels:", labels)
"""
open(f"{TMP}/convert_btxrd.py", "w").write(convert_src)

if DATA_SOURCE == "btxrd":
    if BTXRD_DIR is None:
        found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/Annotations", recursive=True)
                 if os.path.isdir(os.path.join(os.path.dirname(p), "images"))]
        assert found, "Không tìm thấy thư mục BTXRD (images/ + Annotations/) trong /kaggle/input"
        BTXRD_DIR = found[0]
    print("BTXRD_DIR =", BTXRD_DIR)
    if not os.path.isfile(f"{RAW_DS}/dataset.json"):
        !python {TMP}/convert_btxrd.py "{BTXRD_DIR}" "{RAW_DS}" {LABEL_MODE} {MAX_SIZE} {TEST_RATIO} {SEED} {DATASET_NAME}
else:
    # Dữ liệu đã ở định dạng nnU-Net (vd Dataset704_Endovis17 của U-Mamba): DATASET_ID/NAME phải khớp tên thư mục
    !cp -rn {NNUNET_RAW_DIR}/. {RAW_DS}/

!cat {RAW_DS}/dataset.json
for d in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    print(d, len(glob.glob(f"{RAW_DS}/{d}/*")))

## 6. Plan & preprocess (giống README gốc)

In [ ]:
os.chdir(REPO)
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} -c 2d -np 4 --verify_dataset_integrity

import json
plans_path = f"{TMP}/nnUNet_preprocessed/{DATASET_NAME}/nnUNetPlans.json"
plans = json.load(open(plans_path))
if BATCH_SIZE:
    plans["configurations"]["2d"]["batch_size"] = int(BATCH_SIZE)
    json.dump(plans, open(plans_path, "w"), indent=2)
cfg = plans["configurations"]["2d"]
print("patch_size:", cfg["patch_size"], "| batch_size:", cfg["batch_size"])

## 7. Train + Validation
- Gọi `nnUNetv2_train <ID> 2d <FOLD> -tr <TRAINER>` giống script gốc.
- Khi train xong, nnU-Net tự chạy **validation** trên fold val và ghi kết quả vào `fold_X/validation/summary.json`.
- Nếu đã có `checkpoint_latest.pth` (phiên trước bị dừng, hoặc đã đặt `RESUME_DIR`), notebook sẽ tự thêm `--c` để train tiếp.

In [ ]:
os.chdir(REPO)
OUT_DIR  = f"{REPO}/data/nnUNet_results/{DATASET_NAME}/{TRAINER}__nnUNetPlans__2d"
FOLD_DIR = f"{OUT_DIR}/fold_{FOLD}"
resume = "--c" if os.path.isfile(f"{FOLD_DIR}/checkpoint_latest.pth") or os.path.isfile(f"{FOLD_DIR}/checkpoint_final.pth") else ""
print("resume flag:", resume or "(train mới)")
!CUDA_VISIBLE_DEVICES=0 nnUNetv2_train {DATASET_ID} 2d {FOLD} -tr {TRAINER} {resume}

In [ ]:
# Kết quả validation & đường cong huấn luyện
import json
from IPython.display import Image as IPImage, display
summ = f"{FOLD_DIR}/validation/summary.json"
if os.path.isfile(summ):
    s = json.load(open(summ))
    print("Validation mean Dice (foreground):", round(s["foreground_mean"]["Dice"], 4))
    for k, v in s["mean"].items():
        print(f"  class {k}: Dice={v['Dice']:.4f}  IoU={v['IoU']:.4f}")
else:
    print("Chưa có validation/summary.json (train chưa xong?)")
if os.path.isfile(f"{FOLD_DIR}/progress.png"):
    display(IPImage(f"{FOLD_DIR}/progress.png"))

## 8. Inference trên tập test (giống script gốc: `--disable_tta`)

In [ ]:
os.chdir(REPO)
CKPT = next(c for c in ["checkpoint_final.pth", "checkpoint_best.pth", "checkpoint_latest.pth"]
            if os.path.isfile(f"{FOLD_DIR}/{c}"))
PRED_DIR = f"{OUT_DIR}/pred_results_fold{FOLD}"
print("checkpoint:", CKPT)
!CUDA_VISIBLE_DEVICES=0 nnUNetv2_predict -i "{RAW_DS}/imagesTs" -o "{PRED_DIR}" -d {DATASET_ID} -c 2d -tr {TRAINER} -f {FOLD} -chk {CKPT} --disable_tta

## 9. Đánh giá DSC & NSD (dùng script đánh giá gốc cho ảnh PNG 2D)

In [ ]:
os.chdir(REPO)
!python evaluation/endoscopy_DSC_Eval.py --gt_path "{RAW_DS}/labelsTs" --seg_path "{PRED_DIR}" --save_path "{OUT_DIR}/metric_DSC_fold{FOLD}.csv"
!python evaluation/endoscopy_NSD_Eval.py --gt_path "{RAW_DS}/labelsTs" --seg_path "{PRED_DIR}" --save_path "{OUT_DIR}/metric_NSD_fold{FOLD}.csv"

## 10. Trực quan hoá vài kết quả

In [ ]:
import numpy as np, cv2, random
import matplotlib.pyplot as plt
%matplotlib inline

names = sorted(os.listdir(PRED_DIR))
names = [n for n in names if n.endswith(".png")]
random.Random(0).shuffle(names)
k = min(4, len(names))
fig, ax = plt.subplots(k, 3, figsize=(12, 4 * k), squeeze=False)
for i, n in enumerate(names[:k]):
    img = cv2.imread(f"{RAW_DS}/imagesTs/{n[:-4]}_0000.png", cv2.IMREAD_GRAYSCALE)
    gt  = cv2.imread(f"{RAW_DS}/labelsTs/{n}", cv2.IMREAD_UNCHANGED)
    pr  = cv2.imread(f"{PRED_DIR}/{n}", cv2.IMREAD_UNCHANGED)
    ax[i, 0].imshow(img, cmap="gray"); ax[i, 0].set_title(n)
    ax[i, 1].imshow(img, cmap="gray"); ax[i, 1].imshow(np.ma.masked_equal(gt, 0), alpha=0.5, cmap="autumn", vmin=0, vmax=2); ax[i, 1].set_title("Ground truth")
    ax[i, 2].imshow(img, cmap="gray"); ax[i, 2].imshow(np.ma.masked_equal(pr, 0), alpha=0.5, cmap="winter", vmin=0, vmax=2); ax[i, 2].set_title("Prediction")
    for a in ax[i]: a.axis("off")
plt.tight_layout(); plt.show()

## 11. Lưu kết quả
Mọi thứ trong `Swin-UMamba/data/nnUNet_results/` (checkpoint, log, validation, dự đoán, file CSV metric) nằm trong `/kaggle/working` và sẽ được lưu cùng version notebook. Để train tiếp ở phiên sau: thêm output của version này làm input, rồi đặt `RESUME_DIR` trỏ tới thư mục `nnUNet_results` trong đó.

In [ ]:
!du -sh {REPO}/data/nnUNet_results
!ls -la {FOLD_DIR}